# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Goal:** build two retrieval pipelines over the *same* chunked document and compare them.

| | Pipeline 1 – Baseline | Pipeline 2 – Re-Ranking |
|---|---|---|
| Stage 1 | Bi-encoder retrieves Top 10 by cosine similarity | *same* Top 10 from Pipeline 1 |
| Stage 2 | – | Cross-encoder scores all 10 (query, chunk) pairs, re-sorts descending |
| Context for the LLM | Top 3 by cosine | Top 3 by cross-encoder score |
| Output | Answer 1 | Answer 2 (identical prompt template) |

**Document:** *Regulation (EU) 2016/679 – General Data Protection Regulation (GDPR)*, official EUR-Lex PDF (88 pages).
It is a good stress test for retrieval because:
- 173 recitals (non-binding explanations) paraphrase the 99 binding articles, so there are many near-duplicate distractors;
- rules are full of exceptions (*"unless…"*, *"shall not apply to…"*), negations and role-specific obligations (controller vs. processor vs. representative);
- many provisions refer to each other only by number (*"Articles 25 to 39"*).

**Models**
- Bi-encoder: `sentence-transformers/all-MiniLM-L6-v2` (384-d embeddings)
- Cross-encoder: `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Vector store: FAISS `IndexFlatIP` over L2-normalised vectors (inner product = cosine similarity)
- LLM: Google Gemini (`gemini-2.5-flash`, temperature 0)

**Notebook layout**
1. Setup
2. Loading & cleaning the PDF
3. Chunking
4. Pipeline 1 – baseline retrieval
5. Pipeline 2 – cross-encoder re-ranking
6. Answer generation (shared prompt)
7. Failure mode analysis
8. Summary metrics
9. Latency profiling
10. Scaling experiment

## 1. Setup

All tunable settings are kept in one place. The Gemini API key is read from a local `.env` file
(`GEMINI_API_KEY=...`), which is git-ignored – see `.env.example`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import os, re, json, time, random, hashlib, platform
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf                      # PyMuPDF, for PDF text + font information
import faiss                        # vector index
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from dotenv import load_dotenv
from IPython.display import display, Markdown
from transformers.utils import logging as hf_logging

hf_logging.disable_progress_bar()          # hide "Loading weights" bars
pd.set_option("display.max_colwidth", 120)

# ---- configuration ----
PDF_PATH        = Path("data/GDPR_EU_2016_679.pdf")
RESULTS_DIR     = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)

BI_ENCODER_NAME    = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

CHUNK_MAX_WORDS = 150   # ~200 word-piece tokens, stays under MiniLM's 256-token limit
CHUNK_OVERLAP   = 30    # words carried over between consecutive chunks of the same article
TOP_K           = 10    # candidates retrieved by the bi-encoder
TOP_N           = 3     # chunks given to the LLM

load_dotenv()
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

# fixed seeds so that everything except API latency is repeatable
random.seed(0); np.random.seed(0); torch.manual_seed(0)

print("Python  :", platform.python_version())
print("Machine :", platform.platform())
print("CPU     :", platform.processor() or platform.machine(), "|", os.cpu_count(), "logical cores")
print("GPU     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only)")
print("Torch threads:", torch.get_num_threads())

Python  : 3.10.13
Machine : Linux-7.0.0-38-generic-x86_64-with-glibc2.39
CPU     : x86_64 | 8 logical cores
GPU     : none (CPU only)
Torch threads: 4


## 2. Loading & cleaning the PDF

Plain `get_text()` mixes three kinds of noise into the regulation text:
- the running footer on every page (`4.5.2016`, `L 119/12`, `Official Journal of the European Union`, `EN`),
- footnotes (`(1) OJ C 229, 31.7.2012, p. 90.`),
- chapter / section headings.

Instead of guessing with regexes alone, I use the **font information** PyMuPDF exposes:
body text is set in 9.6 pt, footnotes and the `EN` tag in 8.5 pt, article headings (`Article 17`) in italic and
article titles in bold. Each line is kept together with its page number and bold/italic flags.

In [2]:
FOOTER_RE  = re.compile(r"^(\d{1,2}\.\d{1,2}\.\d{4}|L \d+/\d+|Official Journal of the European Union)$")
BODY_SIZE  = 9.0          # anything smaller than this is a footnote / page tag
BOLD, ITALIC = 16, 2      # PyMuPDF span flag bits


def load_lines(path):
    # Return every body-text line as {page, text, bold, italic}.
    lines = []
    for page_no, page in enumerate(pymupdf.open(path), start=1):
        for block in page.get_text("dict")["blocks"]:
            for line in block.get("lines", []):
                spans = [s for s in line["spans"] if s["size"] >= BODY_SIZE]   # drop footnote-sized spans
                text = "".join(s["text"] for s in spans).strip()
                if not text or FOOTER_RE.match(text):                          # drop the running footer
                    continue
                flags = spans[0]["flags"]
                lines.append({"page": page_no, "text": text,
                              "bold": bool(flags & BOLD), "italic": bool(flags & ITALIC)})
    return lines


lines = load_lines(PDF_PATH)
print(f"{pymupdf.open(PDF_PATH).page_count} pages -> {len(lines)} body-text lines")
pd.DataFrame(lines[:8])

88 pages -> 4124 body-text lines


,page,text,bold,italic
0,1,I,False,False
1,1,(Legislative acts),False,True
2,1,REGULATIONS,False,False
3,1,REGULATION (EU) 2016/679 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL,True,False
4,1,of 27 April 2016,True,False
5,1,on the protection of natural persons with regard to the processing of personal data and on the free,True,False
6,1,"movement of such data, and repealing Directive 95/46/EC (General Data Protection Regulation)",True,False
7,1,"Having regard to the Treaty on the Functioning of the European Union, and in particular Article 16 thereof,",False,False


### 2.1 Splitting into recitals and articles

The regulation has two parts:
- **Recitals** `(1)` … `(173)` – the "whereas" explanations (pages 1–31);
- **Articles** `Article 1` … `Article 99` – the binding rules (pages 32–88).

I group lines into one *unit* per recital / article and keep the article title as metadata
(e.g. `Article 33 - Notification of a personal data breach to the supervisory authority`).
A recital number only starts a new recital if it is the next number in sequence, so inline
references like "(4)" do not break the text. Words hyphenated across line breaks are re-joined.

In [3]:
ARTICLE_RE = re.compile(r"^Article (\d+)$")
RECITAL_RE = re.compile(r"^\((\d+)\)\s*(.*)$")


def to_units(lines):
    units, cur = [], None
    in_articles, title_open, next_recital = False, False, 1

    def flush():
        if cur and cur["text"].strip():
            units.append(cur)

    for ln in lines:
        t = ln["text"]
        # the first italic 'Article 1' heading marks the end of the recitals
        if not in_articles and ARTICLE_RE.match(t) and ln["italic"]:
            in_articles = True

        if not in_articles:
            if ln["bold"]:
                continue                                    # title block / headings
            m = RECITAL_RE.match(t)
            if m and int(m.group(1)) == next_recital:
                flush()
                cur = {"section": f"Recital ({next_recital})", "page": ln["page"], "text": m.group(2)}
                next_recital += 1
            elif cur is not None:
                cur["text"] += " " + t
            continue

        m = ARTICLE_RE.match(t)
        if m and ln["italic"]:                              # new article heading
            flush()
            cur = {"section": f"Article {m.group(1)}", "page": ln["page"], "text": ""}
            title_open = True
            continue
        if ln["bold"]:
            if cur is not None and title_open:              # article title, can wrap over 2-3 lines
                sep = " - " if " - " not in cur["section"] else " "
                cur["section"] += sep + t
            continue                                        # chapter / section headings are dropped
        title_open = False
        if cur is not None:
            cur["text"] += " " + t
    flush()

    for u in units:
        txt = re.sub(r"(\w)- (\w)", r"\1\2", u["text"])     # re-join hyphenated words
        u["text"] = re.sub(r"\s+", " ", txt).strip()
    return units


units = to_units(lines)
n_rec = sum(u["section"].startswith("Recital") for u in units)
n_art = sum(u["section"].startswith("Article") for u in units)
print(f"{n_rec} recitals, {n_art} articles")
assert n_rec == 173 and n_art == 99, "structure parsing went wrong"
pd.DataFrame(units)[["section", "page", "text"]].iloc[[0, 172, 173, 205]]

173 recitals, 99 articles


,section,page,text
0,Recital (1),1,The protection of natural persons in relation to the processing of personal data is a fundamental right. Article 8(1...
172,Recital (173),31,This Regulation should apply to all matters concerning the protection of fundamental rights and freedoms vis-àvis th...
173,Article 1 - Subject-matter and objectives,32,1. This Regulation lays down rules relating to the protection of natural persons with regard to the processing of pe...
205,Article 33 - Notification of a personal data breach to the supervisory authority,52,"1. In the case of a personal data breach, the controller shall without undue delay and, where feasible, not later th..."


## 3. Chunking

Each recital/article is split into chunks of at most **150 words** with a **30-word overlap**, cutting on
sentence (or `;` / `:`) boundaries so that numbered paragraphs stay intact. Chunks never cross an article
boundary, so every chunk carries a clean `section` label and page number – useful for checking the LLM's
citations later.

150 words is chosen to fit the bi-encoder: `all-MiniLM-L6-v2` truncates input at 256 word-piece tokens, and
anything past that limit would silently be ignored during embedding. The check below confirms no chunk is
truncated.

This list of chunks is built **once** and is used by both pipelines.

In [4]:
SENT_SPLIT_RE = re.compile(r"(?<=[.;:])\s+(?=[A-Z(0-9])")


def chunk_units(units, max_words=CHUNK_MAX_WORDS, overlap=CHUNK_OVERLAP):
    chunks = []
    for u in units:
        # sentence-like pieces; very long sentences are hard-split with overlap
        pieces = []
        for sent in SENT_SPLIT_RE.split(u["text"]):
            words = sent.split()
            while len(words) > max_words:
                pieces.append(" ".join(words[:max_words]))
                words = words[max_words - overlap:]
            if words:
                pieces.append(" ".join(words))

        # greedily pack pieces into chunks, carrying the last `overlap` words forward
        buf = []
        for piece in pieces:
            if buf and len(" ".join(buf + [piece]).split()) > max_words:
                chunks.append({"section": u["section"], "page": u["page"], "text": " ".join(buf)})
                buf = [" ".join(" ".join(buf).split()[-overlap:])]
            buf.append(piece)
        if buf:
            chunks.append({"section": u["section"], "page": u["page"], "text": " ".join(buf)})

    for i, c in enumerate(chunks):
        c["chunk_id"] = i
    return chunks


chunks = chunk_units(units)          # <- the single shared corpus for both pipelines
chunk_texts = [c["text"] for c in chunks]
chunks_df = pd.DataFrame(chunks).set_index("chunk_id")

word_counts = chunks_df["text"].str.split().str.len()
print(f"{len(chunks)} chunks "
      f"({chunks_df.section.str.startswith('Recital').sum()} from recitals, "
      f"{chunks_df.section.str.startswith('Article').sum()} from articles)")
word_counts.describe().round(1).to_frame("words per chunk").T

565 chunks (258 from recitals, 307 from articles)


,count,mean,std,min,25%,50%,75%,max
words per chunk,565.0,109.5,32.9,22.0,85.0,119.0,137.0,164.0


In [5]:
# Make sure no chunk exceeds the bi-encoder's 256 token window (otherwise its tail is never embedded)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(BI_ENCODER_NAME)
token_counts = pd.Series([len(tok(t)["input_ids"]) for t in chunk_texts])
print("tokens per chunk: mean %.0f | max %d | chunks over 256 tokens: %d"
      % (token_counts.mean(), token_counts.max(), (token_counts > 256).sum()))

# a few sample chunks
chunks_df.loc[[0, 323, 366]]

tokens per chunk: mean 128 | max 196 | chunks over 256 tokens: 0


,section,page,text
chunk_id,,,
0,Recital (1),1,The protection of natural persons in relation to the processing of personal data is a fundamental right. Article 8(1...
323,Article 17 - Right to erasure (‘right to be forgotten’),43,1. The data subject shall have the right to obtain from the controller the erasure of personal data concerning him o...
366,Article 33 - Notification of a personal data breach to the supervisory authority,52,"1. In the case of a personal data breach, the controller shall without undue delay and, where feasible, not later th..."


## 4. Pipeline 1 – Baseline retrieval (bi-encoder)

**(a) Indexing.** Every chunk is encoded *independently* by the bi-encoder into a 384-d vector.
Vectors are L2-normalised, so the inner product computed by `IndexFlatIP` equals cosine similarity.
`IndexFlatIP` is an exact (brute-force) index, which is fine for ~560 vectors.

This indexing work is done once, offline; at query time only the query has to be encoded.

In [6]:
t0 = time.perf_counter()
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device="cpu")
bi_load_s = time.perf_counter() - t0

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(chunk_texts, batch_size=32, normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=False).astype("float32")
index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)
index_build_s = time.perf_counter() - t0

print(f"bi-encoder loaded in {bi_load_s:.2f}s")
print(f"embedded + indexed {index.ntotal} chunks (dim={chunk_emb.shape[1]}) in {index_build_s:.2f}s")
print("vectors normalised:", np.allclose(np.linalg.norm(chunk_emb, axis=1), 1.0, atol=1e-4))

bi-encoder loaded in 7.21s
embedded + indexed 565 chunks (dim=384) in 23.61s
vectors normalised: True


**(b) Retrieval.** The query is embedded with the same model and the 10 nearest chunks by cosine similarity are returned.

**(c) Selection.** The baseline simply takes the first 3 of those 10 – no re-ordering.

In [7]:
def retrieve_top_k(query, k=TOP_K):
    # Bi-encoder retrieval: returns k candidates ordered by cosine similarity.
    q_emb = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q_emb, k)
    return [{"chunk_id": int(cid), "bi_rank": rank, "bi_score": float(s)}
            for rank, (cid, s) in enumerate(zip(ids[0], scores[0]), start=1)]


def baseline_top_n(candidates, n=TOP_N):
    # Pipeline 1: take the Top-n straight from the bi-encoder order.
    return candidates[:n]


demo_q = "How quickly must a personal data breach be notified to the supervisory authority?"
demo_top10 = retrieve_top_k(demo_q)
pd.DataFrame(demo_top10).assign(section=lambda d: d.chunk_id.map(chunks_df.section),
                                text=lambda d: d.chunk_id.map(chunks_df.text).str[:90])

,chunk_id,bi_rank,bi_score,section,text
0,366,1,0.776661,Article 33 - Notification of a personal data breach to the supervisory authority,"1. In the case of a personal data breach, the controller shall without undue delay and, wh"
1,134,2,0.775250,Recital (87),It should be ascertained whether all appropriate technological protection and organisation
2,132,3,0.735003,Recital (85),"unauthorised reversal of pseudonymisation, damage to reputation, loss of confidentiality o"
3,371,4,0.659280,Article 34 - Communication of a personal data breach to the data subject,"would involve disproportionate effort. In such a case, there shall instead be a public com"
4,133,5,0.637348,Recital (86),"The controller should communicate to the data subject a personal data breach, without undu"
5,135,6,0.593405,Recital (88),In setting detailed rules concerning the format and procedures applicable to the notificat
6,369,7,0.591959,Article 34 - Communication of a personal data breach to the data subject,1. When the personal data breach is likely to result in a high risk to the rights and free
7,368,8,0.573150,Article 33 - Notification of a personal data breach to the supervisory authority,phases without undue further delay. 5. The controller shall document any personal data bre
8,367,9,0.573057,Article 33 - Notification of a personal data breach to the supervisory authority,"the nature of the personal data breach including where possible, the categories and approx"
9,510,10,0.541692,Article 70 - Tasks of the Board,"issue guidelines, recommendations and best practices in accordance with point (e) of this"


## 5. Pipeline 2 – Cross-encoder re-ranking

**(a)** Start from exactly the same Top-10 list that Pipeline 1 retrieved (no second retrieval).
**(b)** Feed each `(query, chunk)` pair through the cross-encoder. Unlike the bi-encoder, it reads the query and the
chunk *together* in one transformer pass, so every query token can attend to every chunk token; it outputs a single
relevance logit (higher = more relevant).
**(c)** Sort the 10 candidates by that score, descending, and keep the Top 3.

In [8]:
t0 = time.perf_counter()
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device="cpu")
ce_load_s = time.perf_counter() - t0
print(f"cross-encoder loaded in {ce_load_s:.2f}s, max_length={cross_encoder.max_length}")


def rerank(query, candidates):
    # Score every (query, chunk) pair jointly and re-sort the candidates by cross-encoder score.
    pairs = [(query, chunk_texts[c["chunk_id"]]) for c in candidates]
    scores = cross_encoder.predict(pairs, batch_size=len(pairs), show_progress_bar=False)
    reranked = [{**c, "ce_score": float(s)} for c, s in zip(candidates, scores)]
    reranked.sort(key=lambda c: c["ce_score"], reverse=True)
    for rank, c in enumerate(reranked, start=1):
        c["ce_rank"] = rank
    return reranked


def reranked_top_n(reranked, n=TOP_N):
    # Pipeline 2: Top-n after cross-encoder re-sorting.
    return reranked[:n]


demo_reranked = rerank(demo_q, demo_top10)
(pd.DataFrame(demo_reranked)
   .assign(section=lambda d: d.chunk_id.map(chunks_df.section),
           text=lambda d: d.chunk_id.map(chunks_df.text).str[:80])
   [["ce_rank", "bi_rank", "chunk_id", "ce_score", "bi_score", "section", "text"]])

cross-encoder loaded in 18.97s, max_length=512


,ce_rank,bi_rank,chunk_id,ce_score,bi_score,section,text
0,1,1,366,8.232859,0.776661,Article 33 - Notification of a personal data breach to the supervisory authority,"1. In the case of a personal data breach, the controller shall without undue del"
1,2,3,132,7.903715,0.735003,Recital (85),"unauthorised reversal of pseudonymisation, damage to reputation, loss of confide"
2,3,2,134,6.309049,0.775250,Recital (87),It should be ascertained whether all appropriate technological protection and or
3,4,5,133,4.422630,0.637348,Recital (86),"The controller should communicate to the data subject a personal data breach, wi"
4,5,8,368,3.826641,0.573150,Article 33 - Notification of a personal data breach to the supervisory authority,phases without undue further delay. 5. The controller shall document any persona
5,6,4,371,3.134781,0.659280,Article 34 - Communication of a personal data breach to the data subject,"would involve disproportionate effort. In such a case, there shall instead be a"
6,7,6,135,0.424893,0.593405,Recital (88),In setting detailed rules concerning the format and procedures applicable to the
7,8,7,369,-0.012659,0.591959,Article 34 - Communication of a personal data breach to the data subject,1. When the personal data breach is likely to result in a high risk to the right
8,9,9,367,-1.436821,0.573057,Article 33 - Notification of a personal data breach to the supervisory authority,"the nature of the personal data breach including where possible, the categories"
9,10,10,510,-3.644303,0.541692,Article 70 - Tasks of the Board,"issue guidelines, recommendations and best practices in accordance with point (e"


## 6. Answer generation with a shared prompt

Both pipelines go through the **same** `build_prompt()` function, so the only thing that differs between Answer 1
and Answer 2 is *which* 3 chunks are in the context (and their order).

The prompt tells the model to answer only from the supplied excerpts and to cite them. This matters with a famous
document like the GDPR: without it, the LLM could answer from its own memory and hide retrieval mistakes.

Gemini is called with `temperature=0` and thinking disabled to keep answers as deterministic as possible.
Responses are cached in `results/llm_cache.json` (keyed by a hash of the prompt) together with the latency that was
measured when they were generated. Re-running the notebook therefore reuses the same answers and does not need an API key;
set `USE_LLM_CACHE = False` to force fresh calls.

In [9]:
from google import genai
from google.genai import types

PROMPT_TEMPLATE = '''You are a legal assistant answering questions about the EU General Data Protection Regulation (GDPR).
Answer the question using ONLY the numbered context excerpts below. Do not use outside knowledge.
- Cite the excerpts you rely on, e.g. [1], [2].
- State any conditions, exceptions or limitations that the excerpts attach to the rule.
- If the excerpts do not contain the answer, reply exactly: "Not found in the provided context."
Keep the answer under 120 words.

Context:
{context}

Question: {question}
Answer:'''


def build_prompt(question, selected):
    # Identical template for both pipelines; only the selected chunks differ.
    blocks = []
    for i, c in enumerate(selected, start=1):
        ch = chunks[c["chunk_id"]]
        blocks.append(f"[{i}] ({ch['section']}, page {ch['page']})\n{ch['text']}")
    return PROMPT_TEMPLATE.format(context="\n\n".join(blocks), question=question)


USE_LLM_CACHE = True
CACHE_PATH = RESULTS_DIR / "llm_cache.json"
llm_cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}
_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"]) if os.getenv("GEMINI_API_KEY") else None
MIN_CALL_GAP_S, _last_call = 7.0, 0.0     # free tier allows ~10 requests/minute


def generate(prompt, retries=5):
    # Returns (answer_text, latency_seconds). Latency is the real API round-trip time.
    key = hashlib.sha256((GEMINI_MODEL + prompt).encode()).hexdigest()
    if USE_LLM_CACHE and key in llm_cache:
        hit = llm_cache[key]
        return hit["answer"], hit["latency_s"]
    if _client is None:
        raise RuntimeError("No cached answer and GEMINI_API_KEY is not set")
    global _last_call
    for attempt in range(retries):
        time.sleep(max(0.0, MIN_CALL_GAP_S - (time.perf_counter() - _last_call)))   # stay under the RPM limit
        try:
            _last_call = time.perf_counter()
            t0 = time.perf_counter()
            resp = _client.models.generate_content(
                model=GEMINI_MODEL, contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0, max_output_tokens=400,
                    thinking_config=types.ThinkingConfig(thinking_budget=0)))
            latency = time.perf_counter() - t0
            break
        except Exception as e:                      # rate limit / transient error -> back off
            wait = 2 ** attempt * 5
            print(f"  Gemini error ({type(e).__name__}: {str(e)[:60]}), retrying in {wait}s")
            time.sleep(wait)
    else:
        raise RuntimeError("Gemini call failed repeatedly")
    answer = resp.text.strip()
    llm_cache[key] = {"answer": answer, "latency_s": latency}
    CACHE_PATH.write_text(json.dumps(llm_cache, indent=1))
    return answer, latency


print("LLM:", GEMINI_MODEL, "| cached answers:", len(llm_cache))

LLM: gemini-2.5-flash | cached answers: 16


### 6.1 Running both pipelines for one query

`run_both()` executes the full flow and records timings for each stage:

`query → bi-encoder Top 10 → (P1) first 3 → LLM → Answer 1`
`                         ↘ (P2) cross-encoder re-sort → first 3 → LLM → Answer 2`

In [10]:
def run_both(question):
    t0 = time.perf_counter()
    top10 = retrieve_top_k(question)                 # shared stage 1
    t1 = time.perf_counter()
    reranked = rerank(question, top10)               # pipeline 2 only
    t2 = time.perf_counter()

    p1_ctx, p2_ctx = baseline_top_n(top10), reranked_top_n(reranked)
    answer1, llm1_s = generate(build_prompt(question, p1_ctx))
    answer2, llm2_s = generate(build_prompt(question, p2_ctx))

    return {"question": question, "top10": top10, "reranked": reranked,
            "p1_ids": [c["chunk_id"] for c in p1_ctx], "p2_ids": [c["chunk_id"] for c in p2_ctx],
            "answer1": answer1, "answer2": answer2,
            "timing": {"retrieval_ms": (t1 - t0) * 1e3, "rerank_ms": (t2 - t1) * 1e3,
                       "llm1_ms": llm1_s * 1e3, "llm2_ms": llm2_s * 1e3}}


def rank_table(res, gold_ids=()):
    # Top-10 table showing both rankings side by side; '★' marks gold chunks.
    df = pd.DataFrame(res["reranked"]).sort_values("bi_rank")
    df["gold"] = df.chunk_id.isin(gold_ids).map({True: "★", False: ""})
    df["section"] = df.chunk_id.map(chunks_df.section).str[:60]
    df["page"] = df.chunk_id.map(chunks_df.page)
    df["snippet"] = df.chunk_id.map(chunks_df.text).str[:70] + "…"
    df["in P1 top3"] = df.bi_rank <= TOP_N
    df["in P2 top3"] = df.ce_rank <= TOP_N
    return df[["gold", "chunk_id", "bi_rank", "bi_score", "ce_rank", "ce_score",
               "in P1 top3", "in P2 top3", "section", "page", "snippet"]].reset_index(drop=True)


def show_answers(res):
    display(Markdown(
        "| **Answer 1 – Baseline (bi-encoder Top 3)** | **Answer 2 – Re-ranked (cross-encoder Top 3)** |\n|---|---|\n"
        f"| {res['answer1'].replace(chr(10), '<br>')} | {res['answer2'].replace(chr(10), '<br>')} |\n"
        f"| context chunks: {res['p1_ids']} | context chunks: {res['p2_ids']} |"))


demo = run_both(demo_q)
display(rank_table(demo).style.format({"bi_score": "{:.3f}", "ce_score": "{:+.2f}"}))
show_answers(demo)

,gold,chunk_id,bi_rank,bi_score,ce_rank,ce_score,in P1 top3,in P2 top3,section,page,snippet
0,,366,1,0.777,1,+8.23,True,True,Article 33 - Notification of a personal data breach to the s,52,"1. In the case of a personal data breach, the controller shall without…"
1,,134,2,0.775,3,+6.31,True,True,Recital (87),17,It should be ascertained whether all appropriate technological protect…
2,,132,3,0.735,2,+7.90,True,True,Recital (85),16,"unauthorised reversal of pseudonymisation, damage to reputation, loss …"
3,,371,4,0.659,6,+3.13,False,False,Article 34 - Communication of a personal data breach to the,52,"would involve disproportionate effort. In such a case, there shall ins…"
4,,133,5,0.637,4,+4.42,False,False,Recital (86),17,The controller should communicate to the data subject a personal data …
5,,135,6,0.593,7,+0.42,False,False,Recital (88),17,In setting detailed rules concerning the format and procedures applica…
6,,369,7,0.592,8,-0.01,False,False,Article 34 - Communication of a personal data breach to the,52,1. When the personal data breach is likely to result in a high risk to…
7,,368,8,0.573,5,+3.83,False,False,Article 33 - Notification of a personal data breach to the s,52,phases without undue further delay. 5. The controller shall document a…
8,,367,9,0.573,9,-1.44,False,False,Article 33 - Notification of a personal data breach to the s,52,"the nature of the personal data breach including where possible, the c…"
9,,510,10,0.542,10,-3.64,False,False,Article 70 - Tasks of the Board,76,"issue guidelines, recommendations and best practices in accordance wit…"


| **Answer 1 – Baseline (bi-encoder Top 3)** | **Answer 2 – Re-ranked (cross-encoder Top 3)** |
|---|---|
| A personal data breach must be notified to the supervisory authority without undue delay and, where feasible, not later than 72 hours after the controller becomes aware of it [1], [3]. If the notification is not made within 72 hours, reasons for the delay must accompany it [1], [3]. This notification is not required if the breach is unlikely to result in a risk to the rights and freedoms of natural persons [1], [3]. | A personal data breach must be notified to the supervisory authority without undue delay and, where feasible, not later than 72 hours after the controller becomes aware of it [1], [2]. If the notification is not made within 72 hours, reasons for the delay must accompany it [1], [2]. This notification is not required if the personal data breach is unlikely to result in a risk to the rights and freedoms of natural persons [1], [2]. |
| context chunks: [366, 134, 132] | context chunks: [366, 132, 134] |

## 7. Failure mode analysis (stress tests)

### 7.1 How the failure queries were found
I wrote about 110 candidate questions aimed at the failure types in the assignment, ran both rankings for each, and kept
the ones where the chunk that actually answers the question (the **gold chunk**) sits at bi-encoder rank 4–10
(so the baseline LLM never sees it) but is moved into the Top 3 by the cross-encoder.

Gold chunks are not picked by looking at the rankings. Each one is defined by the **article plus an exact phrase from the
regulation that answers the question**, then resolved to a chunk id. Anyone can check this against the PDF.

| ID | Failure type | What makes it hard |
|---|---|---|
| KT-1 | Keyword trap / distractor | Query word *"liable"* matches the liability/compensation provisions (Art. 82, Recital 146); the answer in Art. 27(5) never uses that word. |
| EX-1 | Exception / corner case | The general breach-notification rule has an *"unless … unlikely to result in a risk"* carve-out inside Art. 33(1). |
| NEG-1 | Negation & constraints | Query asks what is *outside* scope; the answer is the *"does not apply to"* list in Art. 2(2). |
| DOC-1 | Document-specific: recitals outrank the binding article | Recitals 57/47 paraphrase identification issues; the binding constraint *"shall not refuse to act … unless"* sits in Art. 12(2) and is phrased via a cross-reference to Art. 11(2). |

Two **limitation cases** are kept as well, because they show when re-ranking does *not* help:

| ID | What happens |
|---|---|
| LIM-1 | Re-ranking *demotes* the answer: the Art. 14(5) exemption drops out of the Top 3 because recitals about research look more relevant to the cross-encoder. |
| LIM-2 | Recall failure: the fine for breach-notification failures (Art. 83(4)) is written as *"Articles 8, 11, 25 to 39"*, so the bi-encoder never retrieves it into the Top 10, and a re-ranker can only re-order what it is given. |
| OBS-1 | Planned as a synonym trap (*"keep a record"* vs. *"document"*). The Art. 33(5) chunk is promoted #9 → #3, but because of the 30-word chunk overlap the same sentence also ends the previous chunk (bi-encoder #2), so the baseline already had the answer. |

In [11]:
FAILURE_CASES = [
    {"id": "KT-1", "category": "Keyword trap / distractor",
     "query": "Is the representative in the Union liable instead of the controller?",
     "gold_section": "Article 27", "gold_phrase": "without prejudice to legal actions which could be initiated against the controller"},
    {"id": "EX-1", "category": "Exception / corner case",
     "query": "Does every data breach have to be reported to the supervisory authority?",
     "gold_section": "Article 33", "gold_phrase": "unless the personal data breach is unlikely to result in a risk"},
    {"id": "NEG-1", "category": "Negation & constraints",
     "query": "Which activities fall outside the scope of the regulation?",
     "gold_section": "Article 2", "gold_phrase": "This Regulation does not apply to the processing of personal data"},
    {"id": "DOC-1", "category": "Document-specific: recitals outrank binding article",
     "query": "Is a controller allowed to refuse to act on a request when it cannot identify the data subject?",
     "gold_section": "Article 12", "gold_phrase": "shall not refuse to act on the request of the data subject"},
    {"id": "LIM-1", "category": "Limitation: re-ranking demotes the answer",
     "query": "Can scientific researchers skip informing people whose data they reuse?",
     "gold_section": "Article 14", "gold_phrase": "would involve a disproportionate effort, in particular for processing for archiving"},
    {"id": "LIM-2", "category": "Limitation: answer not in Top 10 (recall)",
     "query": "What is the maximum fine for failing to notify a personal data breach?",
     "gold_section": "Article 83", "gold_phrase": "10 000 000 EUR"},
    {"id": "OBS-1", "category": "Observation: chunk overlap already rescues the baseline",
     "query": "Must a company keep a record of every data breach, even ones not reported?",
     "gold_section": "Article 33", "gold_phrase": "shall document any personal data breaches"},
]


def resolve_gold(section, phrase):
    # Chunk ids of `section` whose text contains `phrase`.
    sec_match = chunks_df.section.str.match(re.escape(section) + r"( |$)")
    ids = chunks_df.index[sec_match & chunks_df.text.str.contains(phrase, regex=False)].tolist()
    assert ids, f"gold phrase not found: {section} / {phrase}"
    return ids


for case in FAILURE_CASES:
    case["gold_ids"] = resolve_gold(case["gold_section"], case["gold_phrase"])
pd.DataFrame(FAILURE_CASES)[["id", "category", "gold_section", "gold_ids", "query"]]

,id,category,gold_section,gold_ids,query
0,KT-1,Keyword trap / distractor,Article 27,[347],Is the representative in the Union liable instead of the controller?
1,EX-1,Exception / corner case,Article 33,[366],Does every data breach have to be reported to the supervisory authority?
2,NEG-1,Negation & constraints,Article 2,[259],Which activities fall outside the scope of the regulation?
3,DOC-1,Document-specific: recitals outrank binding article,Article 12,[301],Is a controller allowed to refuse to act on a request when it cannot identify the data subject?
4,LIM-1,Limitation: re-ranking demotes the answer,Article 14,[317],Can scientific researchers skip informing people whose data they reuse?
5,LIM-2,Limitation: answer not in Top 10 (recall),Article 83,[538],What is the maximum fine for failing to notify a personal data breach?
6,OBS-1,Observation: chunk overlap already rescues the baseline,Article 33,"[367, 368]","Must a company keep a record of every data breach, even ones not reported?"


In [12]:
def full_bi_rank(query, chunk_id):
    # Rank of a chunk among *all* chunks by cosine similarity (used when it is outside the Top 10).
    q_emb = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    order = np.argsort(-(chunk_emb @ q_emb[0]))
    return int(np.where(order == chunk_id)[0][0]) + 1


def rank_shift_lines(res, gold_ids):
    by_id = {c["chunk_id"]: c for c in res["reranked"]}
    out = []
    for gid in gold_ids:
        sec = chunks_df.section[gid].split(" - ")[0]
        if gid in by_id:
            c = by_id[gid]
            arrow = "⬆" if c["ce_rank"] < c["bi_rank"] else ("⬇" if c["ce_rank"] > c["bi_rank"] else "=")
            out.append(f"**Chunk {gid} ({sec}): Rank #{c['bi_rank']} → Rank #{c['ce_rank']}** {arrow}")
        else:
            out.append(f"**Chunk {gid} ({sec}): not in the bi-encoder Top 10** "
                       f"(rank #{full_bi_rank(res['question'], gid)} of {len(chunks)}), so the cross-encoder never sees it")
    return out


failure_results = {}
for case in FAILURE_CASES:
    res = run_both(case["query"])
    failure_results[case["id"]] = res
print("ran", len(failure_results), "failure queries")

ran 7 failure queries


### 7.2 Results per failure query
For each query: the Top-10 table with both rankings (★ = gold chunk), the rank shift of the gold chunk, and
Answer 1 vs. Answer 2 side by side.

In [13]:
for case in FAILURE_CASES:
    res = failure_results[case["id"]]
    display(Markdown(f"---\n#### {case['id']} – {case['category']}\n**Query:** *{case['query']}*  \n"
                     f"**Gold:** {case['gold_section']} – \"…{case['gold_phrase']}…\""))
    display(rank_table(res, case["gold_ids"]).style.format({"bi_score": "{:.3f}", "ce_score": "{:+.2f}"}))
    display(Markdown("  \n".join(rank_shift_lines(res, case["gold_ids"]))))
    show_answers(res)

---
#### KT-1 – Keyword trap / distractor
**Query:** *Is the representative in the Union liable instead of the controller?*  
**Gold:** Article 27 – "…without prejudice to legal actions which could be initiated against the controller…"

,gold,chunk_id,bi_rank,bi_score,ce_rank,ce_score,in P1 top3,in P2 top3,section,page,snippet
0,,123,1,0.582,3,+0.80,True,True,Recital (80),15,"to the rights and freedoms of natural persons, taking into account the…"
1,,221,2,0.553,4,+0.10,True,False,Recital (146),27,The controller or processor should compensate any damage which a perso…
2,,533,3,0.547,7,-3.49,True,False,Article 82 - Right to compensation and liability,81,or processor shall be exempt from liability under paragraph 2 if it pr…
3,★,347,4,0.544,2,+3.81,False,True,Article 27 - Representatives of controllers or processors no,48,"the Member States where the data subjects, whose personal data are pro…"
4,,122,5,0.542,1,+4.12,False,True,Recital (80),15,Where a controller or a processor not established in the Union is proc…
5,,222,6,0.539,6,-3.05,False,False,Recital (146),27,acts adopted in accordance with this Regulation and Member State law s…
6,,534,7,0.535,8,-3.61,False,False,Article 82 - Right to compensation and liability,81,"responsible for any damage caused by processing, each controller or pr…"
7,,532,8,0.528,9,-3.89,False,False,Article 82 - Right to compensation and liability,81,1. Any person who has suffered material or non-material damage as a re…
8,,486,9,0.519,10,-11.06,False,False,Article 62 - Joint operations of supervisory authorities,72,they are operating. 5. The Member State in whose territory the damage …
9,,430,10,0.517,5,+0.09,False,False,Article 47 - Binding corporate rules,62,and before the competent courts of the Member States in accordance wit…


**Chunk 347 (Article 27): Rank #4 → Rank #2** ⬆

| **Answer 1 – Baseline (bi-encoder Top 3)** | **Answer 2 – Re-ranked (cross-encoder Top 3)** |
|---|---|
| Not found in the provided context. | No, the designation of a representative does not affect the responsibility or liability of the controller or processor under this Regulation [3]. The representative acts on behalf of the controller or processor and may be addressed by supervisory authorities and data subjects [2], [3]. The designation of a representative is also without prejudice to legal actions against the controller or processor themselves [2]. |
| context chunks: [123, 221, 533] | context chunks: [122, 347, 123] |

---
#### EX-1 – Exception / corner case
**Query:** *Does every data breach have to be reported to the supervisory authority?*  
**Gold:** Article 33 – "…unless the personal data breach is unlikely to result in a risk…"

...

| **Answer 1 – Baseline (bi-encoder Top 3)** | **Answer 2 – Re-ranked (cross-encoder Top 3)** |
|---|---|
| Not found in the provided context. | Not found in the provided context. |
| context chunks: [366, 135, 367] | context chunks: [366, 132, 131] |

---
#### OBS-1 – Observation: chunk overlap already rescues the baseline
**Query:** *Must a company keep a record of every data breach, even ones not reported?*  
**Gold:** Article 33 – "…shall document any personal data breaches…"

,gold,chunk_id,bi_rank,bi_score,ce_rank,ce_score,in P1 top3,in P2 top3,section,page,snippet
0,,135,1,0.566,2,-1.28,True,True,Recital (88),17,In setting detailed rules concerning the format and procedures applica…
1,★,367,2,0.550,5,-2.06,True,False,Article 33 - Notification of a personal data breach to the s,52,"the nature of the personal data breach including where possible, the c…"
2,,131,3,0.546,1,-1.02,True,True,Recital (85),16,"A personal data breach may, if not addressed in an appropriate and tim…"
3,,369,4,0.535,8,-3.93,False,False,Article 34 - Communication of a personal data breach to the,52,1. When the personal data breach is likely to result in a high risk to…
4,,134,5,0.531,6,-2.24,False,False,Recital (87),17,It should be ascertained whether all appropriate technological protect…
5,,371,6,0.529,7,-3.73,False,False,Article 34 - Communication of a personal data breach to the,52,"would involve disproportionate effort. In such a case, there shall ins…"
6,,366,7,0.511,4,-1.96,False,False,Article 33 - Notification of a personal data breach to the s,52,"1. In the case of a personal data breach, the controller shall without…"
7,,91,8,0.486,10,-6.09,False,False,Recital (62),12,"However, it is not necessary to impose the obligation to provide infor…"
8,★,368,9,0.455,3,-1.53,False,True,Article 33 - Notification of a personal data breach to the s,52,phases without undue further delay. 5. The controller shall document a…
9,,510,10,0.454,9,-5.23,False,False,Article 70 - Tasks of the Board,76,"issue guidelines, recommendations and best practices in accordance wit…"


**Chunk 367 (Article 33): Rank #2 → Rank #5** ⬇  
**Chunk 368 (Article 33): Rank #9 → Rank #3** ⬆

| **Answer 1 – Baseline (bi-encoder Top 3)** | **Answer 2 – Re-ranked (cross-encoder Top 3)** |
|---|---|
| Yes, a company must document any personal data breaches, including the facts, effects, and remedial actions taken [2]. This applies even if the breach is not reported. The documentation should consider circumstances like whether data was protected by technical measures [1]. A breach can cause various damages if not addressed timely [3]. | Yes, a company must document any personal data breaches, including the facts, effects, and remedial actions taken [3]. This documentation is required to enable the supervisory authority to verify compliance [3]. The context does not specify any exceptions for breaches that are not reported. |
| context chunks: [135, 367, 131] | context chunks: [131, 135, 368] |

### 7.3 Manual verdicts on the answers
I read each answer pair against the gold text and labelled it as one of: **Correct**, **Abstained** (the model said
the answer was not in the context, so the rule was omitted), **Omitted essential clause**, **Misstated rule** or **Hallucination**.

In [14]:
VERDICTS = {
    "KT-1":  ("Abstained", "Correct",
              "Baseline context = Recital 80, Recital 146 and Art. 82(3), all about liability/compensation, none about the representative. "
              "Re-ranked context contains Art. 27(5): designation is without prejudice to actions against the controller."),
    "EX-1":  ("Abstained", "Correct",
              "Baseline context = Recital 87, Art. 34 (telling data subjects) and Art. 57 (tasks). Answer 2 states the rule *and* the "
              "'unless unlikely to result in a risk' exception from Art. 33(1)."),
    "NEG-1": ("Abstained", "Correct",
              "Baseline context = Art. 57, Recital 73 and Art. 40, all unrelated. Answer 2 lists the Art. 2(2)(a)-(d) exclusions."),
    "DOC-1": ("Omitted essential clause", "Correct",
              "Answer 1 only describes Art. 11 / Recital 57 and misses the binding Art. 12(2) rule: the controller shall NOT refuse "
              "unless it demonstrates it cannot identify the data subject."),
    "LIM-1": ("Correct", "Abstained",
              "Re-ranking hurt: Recital 62 (bi-encoder #1, contains the exemption) and Art. 14(5) both fall out of the Top 3."),
    "LIM-2": ("Abstained", "Abstained",
              "Art. 83(4) is at bi-encoder rank #21, outside the Top 10, so neither pipeline can answer."),
    "OBS-1": ("Correct", "Correct",
              "Both contexts contain the Art. 33(5) sentence (via the overlapping chunk 367 for the baseline)."),
}

## 8. Summary

### 8.1 Rank shifts and answer quality for the failure queries

In [15]:
rows = []
for case in FAILURE_CASES:
    res = failure_results[case["id"]]
    by_id = {c["chunk_id"]: c for c in res["reranked"]}
    shifts = [f"{g}: #{by_id[g]['bi_rank']} → #{by_id[g]['ce_rank']}" if g in by_id
              else f"{g}: #{full_bi_rank(case['query'], g)} (outside Top 10)" for g in case["gold_ids"]]
    v1, v2, note = VERDICTS[case["id"]]
    rows.append({"id": case["id"], "category": case["category"], "query": case["query"],
                 "gold section": case["gold_section"], "gold chunk: rank shift": "; ".join(shifts),
                 "gold in P1 top3": any(g in by_id and by_id[g]["bi_rank"] <= TOP_N for g in case["gold_ids"]),
                 "gold in P2 top3": any(g in by_id and by_id[g]["ce_rank"] <= TOP_N for g in case["gold_ids"]),
                 "Answer 1": v1, "Answer 2": v2})
failure_summary = pd.DataFrame(rows)
failure_summary.to_csv(RESULTS_DIR / "failure_summary.csv", index=False)
failure_summary

,id,category,query,gold section,gold chunk: rank shift,gold in P1 top3,gold in P2 top3,Answer 1,Answer 2
0,KT-1,Keyword trap / distractor,Is the representative in the Union liable instead of the controller?,Article 27,347: #4 → #2,False,True,Abstained,Correct
1,EX-1,Exception / corner case,Does every data breach have to be reported to the supervisory authority?,Article 33,366: #5 → #2,False,True,Abstained,Correct
2,NEG-1,Negation & constraints,Which activities fall outside the scope of the regulation?,Article 2,259: #5 → #1,False,True,Abstained,Correct
3,DOC-1,Document-specific: recitals outrank binding article,Is a controller allowed to refuse to act on a request when it cannot identify the data subject?,Article 12,301: #4 → #1,False,True,Omitted essential clause,Correct
4,LIM-1,Limitation: re-ranking demotes the answer,Can scientific researchers skip informing people whose data they reuse?,Article 14,317: #4 → #9,False,False,Correct,Abstained
5,LIM-2,Limitation: answer not in Top 10 (recall),What is the maximum fine for failing to notify a personal data breach?,Article 83,538: #21 (outside Top 10),False,False,Abstained,Abstained
6,OBS-1,Observation: chunk overlap already rescues the baseline,"Must a company keep a record of every data breach, even ones not reported?",Article 33,367: #2 → #5; 368: #9 → #3,True,True,Correct,Correct


### 8.2 Re-ranking over the whole candidate pool
The failure queries above were *selected* because re-ranking helped (or clearly failed). To get a less biased picture,
the cell below runs **all ~110 candidate questions** I wrote while searching (each with a phrase-defined gold chunk)
through both rankings, without the LLM. The "intended type" column is the failure type I was aiming for when I wrote the question.

Metrics (gold = any chunk containing the gold phrase):
- **Hit@3** – gold chunk is in the Top 3 given to the LLM
- **MRR@10** – mean of 1/rank of the first gold chunk within the Top 10 (0 if absent)
- **Recall@10** – gold chunk is anywhere in the bi-encoder Top 10 (identical for both pipelines, since re-ranking only re-orders)

In [16]:
CANDIDATE_POOL = [
    ('keyword trap', 'What is the age of digital consent for children?', 'not below 13 years'),
    ('keyword trap', 'Can I get my data in a format that lets me move it to another service provider?', 'structured, commonly used and machine-readable'),
    ('keyword trap', 'Can my employer monitor my personal data at work?', 'employment context'),
    ('keyword trap', 'Who appoints the members of a supervisory authority?', 'appointed by means of a transparent procedure'),
    ('keyword trap', 'What happens if hackers steal customer data?', 'without undue delay and, where feasible, not later than 72 hours'),
    ('keyword trap', 'How much compensation can a victim receive for damage caused by unlawful processing?', 'shall have the right to receive compensation'),
    ('keyword trap', 'Can I make the company forget everything about me?', 'right to obtain from the controller the erasure'),
    ('keyword trap', "What penalties can a supervisory authority impose for breaching rules on children's consent?", 'up to 10 000 000 EUR'),
    ('exception', 'Must the controller always inform the individuals affected by a data breach, even if the data were encrypted?', 'shall not be required if any of the following'),
    ('exception', 'Do companies outside the EU always have to appoint a representative in the EU?', 'processing which is occasional'),
    ('exception', 'Is a small company with 50 employees required to keep records of processing?', 'fewer than 250 persons'),
    ('exception', 'Can sensitive data like health data ever be processed?', 'explicit consent to the processing of those personal data for one or more specified purposes'),
    ('exception', "Can a controller refuse a data subject's request?", 'manifestly unfounded or excessive'),
    ('exception', 'Does the right to data portability apply to processing carried out in the public interest?', 'shall not apply to processing necessary for the performance of a task carried out in the public interest'),
    ('exception', 'Can the right to object be overridden by the controller?', 'compelling legitimate grounds'),
    ('exception', 'Can a decision based solely on automated processing be allowed if the person consents?', "is based on the data subject's explicit consent"),
    ('exception', 'When can a controller process data for a new purpose without consent?', 'compatible with the purpose for which the personal data are initially collected'),
    ('negation', 'Which kinds of personal data are prohibited from being processed?', 'Processing of personal data revealing racial or ethnic origin'),
    ('negation', 'Is the data protection officer allowed to take instructions from the controller?', 'does not receive any instructions'),
    ('negation', "Can a processor process personal data without the controller's instructions?", 'shall not process those data except on instructions from the controller'),
    ('negation', 'What processing is the GDPR not applicable to?', 'purely personal or household activity'),
    ('negation', 'Are supervisory authorities not competent to supervise courts?', 'courts acting in their judicial capacity'),
    ('negation', 'When is consent not considered freely given?', 'performance of a contract, including the provision of a service, is conditional on consent'),
    ('negation', 'Can criminal conviction data be processed by anyone other than official authorities?', 'under the control of official authority'),
    ('negation', 'Is a controller forbidden from transferring data to a third country based on a foreign court judgment?', 'Any judgment of a court or tribunal'),
    ('negation', 'Can a processor be penalised for acting outside the instructions of the controller?', 'shall be considered to be a controller in respect of that processing'),
    ('keyword trap', 'Does the GDPR require personal data to be encrypted?', 'the pseudonymisation and encryption of personal data'),
    ('keyword trap', 'When does the GDPR take effect?', 'It shall apply from 25 May 2018'),
    ('keyword trap', 'What is the highest possible fine under the GDPR?', '20 000 000 EUR'),
    ('keyword trap', 'How often must an adequacy decision be reviewed?', 'at least every four years'),
    ('keyword trap', 'How quickly must a processor tell the controller about a data breach?', 'The processor shall notify the controller without undue delay'),
    ('keyword trap', 'Can I get a second copy of my data and does it cost money?', 'any further copies requested by the data subject'),
    ('keyword trap', 'Who decides why and how personal data are processed?', 'determines the purposes and means of the processing'),
    ('keyword trap', 'What makes an agreement to data processing valid?', 'freely given, specific, informed and unambiguous'),
    ('keyword trap', 'Are cookies considered personal data?', 'cookie identifiers'),
    ('keyword trap', 'Is fraud prevention a legitimate reason to process data?', 'preventing fraud'),
    ('keyword trap', 'Can a group of companies share one data protection officer?', 'group of undertakings may appoint a single data protection officer'),
    ('keyword trap', 'Can the data protection officer be an external contractor?', 'on the basis of a service contract'),
    ('keyword trap', 'How long is a data protection certification valid?', 'maximum period of three years'),
    ('keyword trap', 'Who is liable if a sub-processor fails to meet its obligations?', 'initial processor shall remain fully liable'),
    ('keyword trap', 'Does the controller have to tell other companies it shared my data with that it has deleted it?', 'shall communicate any rectification or erasure'),
    ('keyword trap', 'Can a supervisory authority stop a company from processing data altogether?', 'including a ban on processing'),
    ('keyword trap', 'Does the GDPR apply to an American company selling products to people in France?', 'offering of goods or services, irrespective of whether a payment'),
    ('keyword trap', 'What is pseudonymisation?', 'can no longer be attributed to a specific data subject without the use of additional information'),
    ('exception', 'If stolen data were encrypted, do the affected users need to be told?', 'unintelligible'),
    ('exception', 'Can a bank refuse to delete my data because a law requires it to keep records?', 'for compliance with a legal obligation which requires processing'),
    ('exception', 'Can a company continue sending me marketing after I object?', 'shall no longer be processed for such purposes'),
    ('exception', 'Do courts need to appoint a data protection officer?', 'except for courts acting in their judicial capacity'),
    ('exception', 'Can public authorities be fined under the GDPR?', 'administrative fines may be imposed on public authorities'),
    ('exception', 'Can a government agency rely on legitimate interests to process data?', 'shall not apply to processing carried out by public authorities in the performance of their tasks'),
    ('exception', 'How long does a company have to answer my request and can it take longer?', 'extended by two further months'),
    ('exception', 'Can data be transferred abroad if the person explicitly agrees?', 'explicitly consented to the proposed transfer'),
    ('exception', 'Can a company transfer data abroad based only on its own legitimate interests?', 'not repetitive, concerns only a limited number'),
    ('exception', 'Can a controller avoid paying compensation for damage?', 'exempt from liability'),
    ('exception', 'When is a processor liable for damage caused by processing?', 'specifically directed to processors'),
    ('exception', 'Can sensitive data that a person has posted publicly be processed?', 'manifestly made public by the data subject'),
    ('exception', 'Does the controller need to tell me information I already know?', 'already has the information'),
    ('exception', 'Can my right to access my data be limited to protect other people?', 'shall not adversely affect the rights and freedoms of others'),
    ('negation', 'Does the GDPR protect the data of dead people?', 'does not apply to the personal data of deceased persons'),
    ('negation', 'Does the regulation protect data about companies?', 'concerns legal persons'),
    ('negation', 'Does the GDPR apply to anonymised data?', 'does not therefore concern the processing of such anonymous information'),
    ('negation', 'Is a processor allowed to hire a sub-processor without written authorisation?', 'shall not engage another processor without prior specific or general written authorisation'),
    ('negation', 'Can the data protection officer have other duties?', 'conflict of interests'),
    ('negation', 'Is a controller allowed to refuse to act on a request when it cannot identify the data subject?', 'shall not refuse to act on the request of the data subject'),
    ('negation', "Can processing of children's data be based on consent of a 12 year old?", 'not below 13 years'),
    ('negation', 'Can personal data be kept forever?', 'no longer than is necessary'),
    ('negation', 'Can the Commission give instructions to the European Data Protection Board?', 'shall act independently'),
    ('negation', 'Which supervisory authority members are not allowed to take instructions?', 'free from external influence'),
    ('keyword trap', 'What is the maximum fine for failing to notify a personal data breach?', '10 000 000 EUR'),
    ('keyword trap', 'What fine applies if a company does not report a data breach in time?', '10 000 000 EUR'),
    ('keyword trap', "How large can the fine be for violating a data subject's right of access?", '20 000 000 EUR'),
    ('keyword trap', "What is the penalty for ignoring a data subject's erasure request?", '20 000 000 EUR'),
    ('keyword trap', 'What is the fine for not appointing a data protection officer?', '10 000 000 EUR'),
    ('keyword trap', 'What is meant by restriction of processing?', 'marking of stored personal data'),
    ('keyword trap', 'What counts as a personal data breach?', 'breach of security leading to the accidental or unlawful destruction'),
    ('keyword trap', 'Does every company need a data protection officer?', 'regular and systematic monitoring of data subjects on a large scale'),
    ('keyword trap', 'What is an enterprise under the GDPR?', 'engaged in an economic activity, irrespective of its legal form'),
    ('keyword trap', 'Who counts as a recipient of personal data?', 'to which the personal data are disclosed, whether a third party or not'),
    ('keyword trap', 'Can my employer read my work emails?', 'employment context'),
    ('keyword trap', 'What is the deadline for answering a data subject access request?', 'within one month of receipt of the request'),
    ('keyword trap', 'Does a data subject have to pay to lodge a complaint with the supervisory authority?', 'free of charge for the data subject'),
    ('keyword trap', 'Can member states set rules for processing a national identification number?', 'national identification number'),
    ('keyword trap', 'Within how many months must the Board adopt a decision in a dispute between authorities, and can it be extended?', 'two-thirds majority of the members of the Board'),
    ('keyword trap', 'Within what period must a supervisory authority reply to a mutual assistance request from another authority?', 'no later than one month after receiving the request'),
    ('keyword trap', 'Can personal data be processed for journalism without consent?', 'journalistic purposes'),
    ('keyword trap', 'Is the representative in the Union liable instead of the controller?', 'without prejudice to legal actions which could be initiated against the controller'),
    ('keyword trap', 'Is the lead supervisory authority obliged to share its draft decision with other authorities?', 'submit a draft decision to the other supervisory authorities concerned'),
    ('keyword trap', 'Who is the lead supervisory authority for a company with offices in several EU countries?', 'main establishment or of the single establishment'),
    ('keyword trap', 'Can I ask a company to stop using my data while it checks whether the data is correct?', 'accuracy of the personal data is contested'),
    ('keyword trap', 'Can I make a company freeze my data instead of deleting it?', 'opposes the erasure'),
    ('keyword trap', 'Is a non-profit allowed to sue on my behalf?', 'not-for-profit body'),
    ('keyword trap', 'Can a consumer organisation file a complaint without my mandate?', "independently of a data subject's mandate"),
    ('keyword trap', 'Does CCTV monitoring of a shopping mall require an impact assessment?', 'publicly accessible area'),
    ('keyword trap', 'Does a hospital need an impact assessment for processing patient records at scale?', 'processing on a large scale of special categories'),
    ('keyword trap', 'Who is the boss of the data protection officer?', 'highest management level'),
    ('keyword trap', "Who pays for the data protection officer's training?", 'maintain his or her expert knowledge'),
    ('keyword trap', 'What must a company check before using data for a different purpose than originally collected?', 'any link between the purposes'),
    ('keyword trap', 'Can I complain in my home country if the company is based in another EU country?', 'habitual residence, place of work'),
    ('keyword trap', 'Is a company liable if its processor causes damage?', 'shall be held liable for the entire damage'),
    ('keyword trap', 'Does the representative of a non-EU company have to keep records of processing?', "the controller's representative, shall maintain a record"),
    ('keyword trap', 'What happens to the data when the contract with the processor ends?', 'deletes or returns all the personal data'),
    ('keyword trap', 'Does a company have to tell me how long it will keep my data?', 'the period for which the personal data will be stored'),
    ('keyword trap', 'Does a company have to tell me where it got my data from?', 'from which source the personal data originate'),
    ('keyword trap', 'Do I have to give my data to a company in order to sign a contract with it?', 'statutory or contractual requirement'),
    ('keyword trap', 'Must the controller tell me if it plans to send my data abroad?', 'intends to transfer personal data to a third country'),
    ('keyword trap', 'Is the processor liable instead of the controller when it breaks the contract?', 'shall be considered to be a controller in respect of that processing'),
    ('keyword trap', 'Who has to keep processing contracts in writing?', 'shall be in writing, including in electronic form'),
    ('keyword trap', 'Does the processor have to help the controller answer data subject requests?', 'assists the controller by appropriate technical and organisational measures'),
]


pool_rows = []
for intended, q, phrase in CANDIDATE_POOL:
    gold = set(chunks_df.index[chunks_df.text.str.lower().str.contains(phrase.lower(), regex=False)])
    reranked = rerank(q, retrieve_top_k(q))
    bi_hits = [c["bi_rank"] for c in reranked if c["chunk_id"] in gold]
    ce_hits = [c["ce_rank"] for c in reranked if c["chunk_id"] in gold]
    pool_rows.append({"intended type": intended, "query": q,
                      "bi_rank": min(bi_hits) if bi_hits else None, "ce_rank": min(ce_hits) if ce_hits else None})
pool_df = pd.DataFrame(pool_rows)
pool_df["recall@10"] = pool_df.bi_rank.notna()
pool_df["P1 hit@3"]  = pool_df.bi_rank.le(TOP_N)
pool_df["P2 hit@3"]  = pool_df.ce_rank.le(TOP_N)
pool_df["P1 RR"] = (1 / pool_df.bi_rank).fillna(0)
pool_df["P2 RR"] = (1 / pool_df.ce_rank).fillna(0)
pool_df.to_csv(RESULTS_DIR / "candidate_pool_ranks.csv", index=False)


def summarise(df):
    return pd.Series({"queries": len(df),
                      "Recall@10": df["recall@10"].mean(),
                      "P1 Hit@3": df["P1 hit@3"].mean(), "P2 Hit@3": df["P2 hit@3"].mean(),
                      "P1 MRR@10": df["P1 RR"].mean(), "P2 MRR@10": df["P2 RR"].mean(),
                      "promoted into top3": int((~df["P1 hit@3"] & df["P2 hit@3"]).sum()),
                      "pushed out of top3": int((df["P1 hit@3"] & ~df["P2 hit@3"]).sum())})


pool_summary = pd.concat([pool_df.groupby("intended type").apply(summarise).T,
                          summarise(pool_df).rename("ALL")], axis=1).T
pool_summary.to_csv(RESULTS_DIR / "candidate_pool_summary.csv")
pool_summary.style.format({c: "{:.2f}" for c in ["Recall@10", "P1 Hit@3", "P2 Hit@3", "P1 MRR@10", "P2 MRR@10"]}
                          | {c: "{:.0f}" for c in ["queries", "promoted into top3", "pushed out of top3"]})

,queries,Recall@10,P1 Hit@3,P2 Hit@3,P1 MRR@10,P2 MRR@10,promoted into top3,pushed out of top3
exception,23,0.57,0.52,0.52,0.37,0.48,0,0
keyword trap,66,0.62,0.47,0.48,0.38,0.42,4,3
negation,19,0.84,0.68,0.74,0.59,0.68,2,1
ALL,108,0.65,0.52,0.54,0.42,0.48,6,4


In [17]:
# Where did the gold chunk move? (only queries whose gold chunk is inside the Top 10)
moved = pool_df.dropna(subset=["bi_rank"]).astype({"bi_rank": int, "ce_rank": int})
shift = pd.crosstab(moved.bi_rank.rename("bi-encoder rank"), moved.ce_rank.rename("cross-encoder rank"))
shift

cross-encoder rank,1,2,3,4,5,6,8,10
bi-encoder rank,,,,,,,,
1,27,1,1,0,0,1,0,1
2,12,5,0,1,0,0,0,0
3,3,3,0,0,0,0,1,0
4,1,2,2,0,0,0,0,0
5,0,0,1,0,1,1,0,0
6,0,0,0,1,0,0,0,0
7,0,0,0,0,0,0,0,1
8,0,0,0,0,1,0,1,1
9,0,0,0,0,1,0,0,0


## 9. Latency profiling

Each retrieval stage is timed with `time.perf_counter()` on CPU, after a few warm-up runs (the first calls include
lazy initialisation). Each failure query is repeated 20 times.

| Stage | Pipeline 1 | Pipeline 2 |
|---|---|---|
| query embedding (bi-encoder) | ✔ | ✔ |
| FAISS search, Top 10 | ✔ | ✔ |
| cross-encoder scoring of 10 pairs + sort | – | ✔ |
| LLM generation (Gemini API) | ✔ | ✔ |

LLM latency is a network round-trip to Gemini, so it is reported separately: it is much larger and noisier than the
local retrieval steps. It is the time measured when each answer was generated, stored in the cache.
The pipeline comparison uses **medians**, because with only 7 LLM calls per pipeline one slow API response would
otherwise decide which pipeline looks faster end-to-end.

In [18]:
N_REPEATS, N_WARMUP = 20, 5
queries = [c["query"] for c in FAILURE_CASES]

for q in queries[:N_WARMUP]:                       # warm-up
    rerank(q, retrieve_top_k(q))

timings = {"query embedding": [], "FAISS search (top 10)": [], "cross-encoder (10 pairs) + sort": []}
for _ in range(N_REPEATS):
    for q in queries:
        t0 = time.perf_counter()
        q_emb = bi_encoder.encode([q], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        t1 = time.perf_counter()
        scores, ids = index.search(q_emb, TOP_K)
        t2 = time.perf_counter()
        cands = [{"chunk_id": int(i), "bi_rank": r, "bi_score": float(s)}
                 for r, (i, s) in enumerate(zip(ids[0], scores[0]), start=1)]
        rerank(q, cands)
        t3 = time.perf_counter()
        timings["query embedding"].append((t1 - t0) * 1e3)
        timings["FAISS search (top 10)"].append((t2 - t1) * 1e3)
        timings["cross-encoder (10 pairs) + sort"].append((t3 - t2) * 1e3)

t = {k: np.array(v) for k, v in timings.items()}
t["Pipeline 1 retrieval (embed + search)"] = t["query embedding"] + t["FAISS search (top 10)"]
t["Pipeline 2 retrieval (embed + search + re-rank)"] = t["Pipeline 1 retrieval (embed + search)"] + t["cross-encoder (10 pairs) + sort"]

llm1 = np.array([failure_results[c["id"]]["timing"]["llm1_ms"] for c in FAILURE_CASES])
llm2 = np.array([failure_results[c["id"]]["timing"]["llm2_ms"] for c in FAILURE_CASES])
t["LLM generation – Answer 1"] = llm1
t["LLM generation – Answer 2"] = llm2


def stats(x):
    return {"n": len(x), "mean (ms)": x.mean(), "median (ms)": np.median(x), "p95 (ms)": np.percentile(x, 95)}


latency = pd.DataFrame({k: stats(v) for k, v in t.items()}).T
latency.to_csv(RESULTS_DIR / "latency_stages.csv")
latency.style.format({"n": "{:.0f}", "mean (ms)": "{:.1f}", "median (ms)": "{:.1f}", "p95 (ms)": "{:.1f}"})

,n,mean (ms),median (ms),p95 (ms)
query embedding,140,18.0,17.7,21.2
FAISS search (top 10),140,0.1,0.1,0.2
cross-encoder (10 pairs) + sort,140,550.3,547.4,611.6
Pipeline 1 retrieval (embed + search),140,18.1,17.8,21.4
Pipeline 2 retrieval (embed + search + re-rank),140,568.5,565.0,628.6
LLM generation – Answer 1,7,2678.7,1292.1,8285.1
LLM generation – Answer 2,7,1251.2,1261.5,1507.8


In [19]:
# medians: a single slow Gemini call would otherwise dominate the mean of only 7 LLM samples
p1 = latency.loc["Pipeline 1 retrieval (embed + search)", "median (ms)"]
p2 = latency.loc["Pipeline 2 retrieval (embed + search + re-rank)", "median (ms)"]
l1, l2 = np.median(llm1), np.median(llm2)

comparison = pd.DataFrame({
    "Pipeline 1 – Baseline":   {"retrieval (ms)": p1, "re-ranking (ms)": 0.0, "LLM (ms)": l1, "end-to-end (ms)": p1 + l1},
    "Pipeline 2 – Re-ranking": {"retrieval (ms)": p1, "re-ranking (ms)": p2 - p1, "LLM (ms)": l2, "end-to-end (ms)": p2 + l2},
}).T
comparison.to_csv(RESULTS_DIR / "latency_pipelines.csv")
display(comparison.style.format("{:.1f}"))
print(f"re-ranking makes retrieval {p2 / p1:.1f}x slower (+{p2 - p1:.1f} ms), "
      f"but adds only {100 * (p2 - p1) / (p1 + l1):.1f}% to the end-to-end time.")

one_off = pd.Series({"load bi-encoder": bi_load_s, "embed + index 565 chunks": index_build_s,
                     "load cross-encoder": ce_load_s}, name="one-off cost (s)")
one_off.to_frame().style.format("{:.2f}")

,retrieval (ms),re-ranking (ms),LLM (ms),end-to-end (ms)
Pipeline 1 – Baseline,17.8,0.0,1292.1,1310.0
Pipeline 2 – Re-ranking,17.8,547.2,1261.5,1826.5


re-ranking makes retrieval 31.7x slower (+547.2 ms), but adds only 41.8% to the end-to-end time.


,one-off cost (s)
load bi-encoder,7.21
embed + index 565 chunks,23.61
load cross-encoder,18.97
